In [98]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

In [99]:
data = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset 2.csv')
data.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [100]:
print(data.columns.tolist())

['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [101]:
#drop non numeric cols
data_clean = data.select_dtypes(include=[np.number]).dropna()

#set target 
X = data_clean.drop(columns=['label'])
y = data_clean['label']

#shows correlation between two features that is above .9 
corr = X.corr()
high_corr = [(col1, col2) for col1 in corr.columns 
             for col2 in corr.columns 
             if col1 < col2 and abs(corr[col1][col2]) > 0.9]

for pair in high_corr:
    print(f"{pair[0]} - {pair[1]}")

NoOfLettersInURL - URLLength
DomainTitleMatchScore - URLTitleMatchScore


In [102]:
# check how correlated each feature is with the label
# high correlation with the label = potential data leakage
# the model would be learning from features that essentially encode the answer
label_corr = data_clean.corr()['label'].drop('label')
print(label_corr.abs().sort_values(ascending=False).to_string())

URLSimilarityIndex            0.860358
HasSocialNet                  0.784255
HasCopyrightInfo              0.743358
HasDescription                0.690232
IsHTTPS                       0.609132
DomainTitleMatchScore         0.584905
HasSubmitButton               0.578561
IsResponsive                  0.548608
URLTitleMatchScore            0.539419
SpacialCharRatioInURL         0.533537
HasHiddenFields               0.507731
HasFavicon                    0.493711
URLCharProb                   0.469749
CharContinuationRate          0.467735
HasTitle                      0.459725
DegitRatioInURL               0.432032
Robots                        0.392620
NoOfJS                        0.373500
LetterRatioInURL              0.367794
Pay                           0.359747
NoOfOtherSpecialCharsInURL    0.358891
NoOfSelfRef                   0.316211
DomainLength                  0.283152
NoOfImage                     0.274658
LineOfCode                    0.272257
NoOfExternalRef          

In [103]:
# drop one from each highly inter-correlated pair
cols_to_drop_corr = [pair[0] for pair in high_corr]
print(f"Dropping correlated features: {cols_to_drop_corr}")
X = X.drop(columns=cols_to_drop_corr)

# drop features with label correlation (data leakage)
threshold = 0.25 #this was all trial and error, this threshold is not statistically calculated
leaky_cols = label_corr[label_corr.abs() >= threshold].index.tolist()
leaky_cols = [col for col in leaky_cols if col in X.columns]

print(f"Dropping leaky features: {leaky_cols}")
X = X.drop(columns=leaky_cols)

Dropping correlated features: ['NoOfLettersInURL', 'DomainTitleMatchScore']
Dropping leaky features: ['DomainLength', 'URLSimilarityIndex', 'CharContinuationRate', 'URLCharProb', 'LetterRatioInURL', 'DegitRatioInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'HasTitle', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'HasDescription', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'Pay', 'HasCopyrightInfo', 'NoOfImage', 'NoOfJS', 'NoOfSelfRef', 'NoOfExternalRef']


In [104]:
X = X.drop(columns=['NoOfCSS']) #I had to hard code this one because it was causing a leak but was way below my threshold
#I checked this cols correlation after training the model at the end of the notebook, 
#saw it was maybe causing a leak and then retrained the model

In [105]:
#I add k-fold CV to make sure lucky splits weren't causing our high accuracies 
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(solver='lbfgs', random_state=42, max_iter=1000))])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=kf, scoring='accuracy')

print(scores)


[0.85913611 0.86201997 0.8590937  0.86119299 0.85570093]


In [106]:
#split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [107]:
#scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#train and fit model
log_reg = LogisticRegression(solver='lbfgs', random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

In [ ]:
#test model
y_pred = log_reg.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

In [ ]:

cm = confusion_matrix(y_test, y_pred)
plt.figure()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()


In [ ]:
#This was to check the importance of features in the model, there was a clear data leakage, might still be one but it looks a lot better!
importance = pd.Series(abs(log_reg.coef_[0]), index=X.columns)
importance.sort_values(ascending=False).head(10)